In [2]:
# Cell 1 – Setup path + imports
import sys
from pathlib import Path

# === CRITICAL: Add project root to Python path ===
project_root = Path.cwd().parent          # This goes one level up to AI-RAG-BOT
sys.path.append(str(project_root))

print("✅ Project root added:", project_root)

# Now normal imports will work
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS   # ← we are using FAISS now

from src.config import (
    DATA_RAW_DIR, FAISS_PATH, EMBEDDING_MODEL,
    CHUNK_SIZE, CHUNK_OVERLAP
)

print("Raw data folder:", DATA_RAW_DIR)
print("FAISS index will be saved to:", FAISS_PATH)

✅ Project root added: c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT
Raw data folder: c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\data\raw
FAISS index will be saved to: c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\faiss_index


In [3]:
# Cell 2 – load all PDFs
docs = []

for pdf_path in DATA_RAW_DIR.glob("*.pdf"):
    print(f"Loading {pdf_path.name} ...")
    loader = PyMuPDFLoader(str(pdf_path))
    loaded = loader.load()
    for doc in loaded:
        doc.metadata["source_file"] = pdf_path.name
        doc.metadata["page"] = doc.metadata.get("page", 0) + 1
    docs.extend(loaded)

print(f"Total pages loaded: {len(docs)}")

Loading EAL Domestic Travel Policy - Non Sales 2017.pdf ...
Loading EAL PF SOP.pdf ...
Loading EAL-Signed FS & Auditor Report FY 16-17.pdf ...
Loading Final Draft Annual KRA 2021-22 - 24.04.2022.pdf ...
Loading Group Leave Policy 1.pdf ...


c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\rag-venv\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:299: UserWarning: Warning: Empty content on page 0 of document c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\data\raw\EAL-Signed FS & Auditor Report FY 16-17.pdf
  warnings.warn(
c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\rag-venv\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:299: UserWarning: Warning: Empty content on page 1 of document c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\data\raw\EAL-Signed FS & Auditor Report FY 16-17.pdf
  warnings.warn(
c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\rag-venv\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:299: UserWarning: Warning: Empty content on page 2 of document c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\data\raw

Loading Major Working Areas & Development 2017-2022.pdf ...
Loading MEDICLAIM 2022-23.pdf ...
Loading Standalone FS 17-18.pdf ...
Total pages loaded: 156


c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\rag-venv\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:299: UserWarning: Warning: Empty content on page 0 of document c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\data\raw\Standalone FS 17-18.pdf
  warnings.warn(
c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\rag-venv\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:299: UserWarning: Warning: Empty content on page 1 of document c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\data\raw\Standalone FS 17-18.pdf
  warnings.warn(
c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\rag-venv\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:299: UserWarning: Warning: Empty content on page 2 of document c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\data\raw\Standalone FS 17-18.pdf
  warnings.warn

In [4]:
# Cell 3 – chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
    add_start_index=True
)

chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

Created 130 chunks


In [7]:
# Cell 4 – embeddings + FAISS (no build issues)
from langchain_community.vectorstores import FAISS
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Save to disk
vectorstore.save_local(str(FAISS_PATH))
print(f"✅ FAISS index created and saved to {FAISS_PATH}")

✅ FAISS index created and saved to c:\Users\krash\OneDrive\Desktop\ai-audit-engineer-portfolio\AI-RAG-BOT\faiss_index


In [8]:
# Cell 5 – quick retrieval test
query = "What is the approval limit for related party transactions?"
retrieved = vectorstore.similarity_search(query, k=4)

for i, doc in enumerate(retrieved, 1):
    print(f"\nResult {i}:")
    print(f"Score: {doc.metadata.get('score', 'n/a')}")
    print(f"File: {doc.metadata['source_file']} | Page {doc.metadata['page']}")
    print(doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content)
    print("-"*80)


Result 1:
Score: n/a
File: EAL Domestic Travel Policy - Non Sales 2017.pdf | Page 8
EMAMI AGROTECH LIMITED 
                          DOMESTIC TRAVEL POLICY – NON SALES 
 
 
 
Policy Name 
Date of Issue 
Recommended By 
Approved By 
Domestic Travel Policy- 
Non - Sales Function 
01.11.2017 
AVP  HR 
CEO 
 
expense of personal nature shall be reimbursed. Supporting of Daily Expense is mandatory if 
the amount exceeds ₹ 300. 
d) 
In case two or more employees travelling together de...
--------------------------------------------------------------------------------

Result 2:
Score: n/a
File: EAL Domestic Travel Policy - Non Sales 2017.pdf | Page 8
a) 
The Company will reimburse for the actual cost of entertainment for customers, guests and 
suppliers when it is for the benefit of the Company, a valid business purpose exists and it 
reflects. The cost of such entertainment for customers, guests and suppliers should not exceed 
₹ 750 per person (including taxes). Any expense beyond this l